In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Parse timestamp columns to UTC datetimes (Unix seconds -> utc)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# 1) Total number of rows for each file
metric_rows = metric.shape[0]
trace_rows = trace.shape[0]
log_rows = log.shape[0]
error_rows = error_logs.shape[0]

totals = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv'],
    'rows': [metric_rows, trace_rows, log_rows, error_rows]
})

# 2) metric.csv aggregated counts by (cmdb_id, kpi_name), top 20
metric_agg = (
    metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .sort_values('count', ascending=False)
    .head(20)
)

# 3) trace.csv aggregated counts by (cmdb_id, trace_name), top 20
trace_agg = (
    trace.groupby(['cmdb_id', 'trace_name'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .sort_values('count', ascending=False)
    .head(20)
)

# 4) log.csv aggregated counts by (cmdb_id, log_name), top 20
log_agg = (
    log.groupby(['cmdb_id', 'log_name'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .sort_values('count', ascending=False)
    .head(20)
)

# 5) error_logs.csv: top up to 20 cmdb_id counts and earliest/latest timestamp
error_top = (
    error_logs.groupby('cmdb_id', as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .sort_values('count', ascending=False)
    .head(20)
)

error_time_range = pd.DataFrame({
    'earliest_utc': [error_logs['timestamp'].min()],
    'latest_utc': [error_logs['timestamp'].max()]
})

# Display compact outputs (variables separated by commas as required)
totals, metric_agg, trace_agg, log_agg, error_top, error_time_range

```
Out[1]:
```
summary = (
    "Summary of telemetry files:\n\n"
    "- Row counts: metric.csv=1800, trace.csv=3800, log.csv=460, error_logs.csv=0.\n\n"
    "- metric.csv aggregates: top groups show adservice, cartservice, and checkoutservice (and others) "
    "with KPIs like cpu, diskio, latency-50, latency-90, mem, socket, workload each appearing 25 times — "
    "telemetry is uniformly present across many services/KPIs.\n\n"
    "- trace.csv aggregates: checkoutservice dominates the trace entries; many trace metrics (e.g. from_*/to_* duration_mean/p95/error_rate/row_count) each appear 25 times.\n\n"
    "- log.csv aggregates: most services have log.error_count and log.row_count entries with count 25; redis has only 5 entries.\n\n"
    "- error_logs.csv: empty (no rows), so there are no recorded error-log timestamps (earliest/latest are NaT).\n\n"
    "Interpretation: telemetry is consistently sampled across metrics and logs; traces are concentrated on checkoutservice; there are no recorded error log entries to inspect."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file  rows
0      metric.csv  1800
1       trace.csv  3800
2         log.csv   460
3  error_logs.csv     0,             cmdb_id    kpi_name  count
0         adservice         cpu     25
1         adservice      diskio     25
2         adservice  latency-50     25
3         adservice  latency-90     25
4         adservice         mem     25
5         adservice      socket     25
6         adservice    workload     25
7       cartservice         cpu     25
8       cartservice      diskio     25
9       cartservice  latency-50     25
10      cartservice  latency-90     25
11      cartservice         mem     25
12      cartservice      socket     25
13      cartservice    workload     25
14  checkoutservice         cpu     25
15  checkoutservice      diskio     25
16  checkoutservice  latency-50     25
17  checkoutservice  latency-90     25
18  checkoutservice         mem     25
19  checkoutservice      socket     25,             cmdb_id                                trace_name  count
0   checkoutservice  trace.from_checkoutservice.duration_mean     25
1   checkoutservice   trace.from_checkoutservice.duration_p95     25
2   checkoutservice     trace.from_checkoutservice.error_rate     25
3   checkoutservice      trace.from_checkoutservice.row_count     25
4   checkoutservice  trace.from_frontendservice.duration_mean     25
5   checkoutservice   trace.from_frontendservice.duration_p95     25
6   checkoutservice     trace.from_frontendservice.error_rate     25
7   checkoutservice      trace.from_frontendservice.row_count     25
8   checkoutservice             trace.from_root.duration_mean     25
9   checkoutservice              trace.from_root.duration_p95     25
10  checkoutservice                trace.from_root.error_rate     25
11  checkoutservice                 trace.from_root.row_count     25
12  checkoutservice    trace.to_checkoutservice.duration_mean     25
13  checkoutservice     trace.to_checkoutservice.duration_p95     25
14  checkoutservice       trace.to_checkoutservice.error_rate     25
15  checkoutservice        trace.to_checkoutservice.row_count     25
16  checkoutservice    trace.to_currencyservice.duration_mean     25
17  checkoutservice     trace.to_currencyservice.duration_p95     25
18  checkoutservice       trace.to_currencyservice.error_rate     25
19  checkoutservice        trace.to_currencyservice.row_count     25,                   cmdb_id         log_name  count
0               adservice  log.error_count     25
1               adservice    log.row_count     25
2             cartservice  log.error_count     25
3             cartservice    log.row_count     25
4         checkoutservice  log.error_count     25
5         checkoutservice    log.row_count     25
6         currencyservice  log.error_count     25
7         currencyservice    log.row_count     25
8            emailservice  log.error_count     25
9            emailservice    log.row_count     25
10               frontend  log.error_count     25
11               frontend    log.row_count     25
12         paymentservice  log.error_count     25
13         paymentservice    log.row_count     25
14  recommendationservice  log.error_count     25
15  recommendationservice    log.row_count     25
19        shippingservice    log.row_count     25
18        shippingservice  log.error_count     25
17                  redis    log.row_count      5
16                  redis  log.error_count      5, Empty DataFrame
Columns: [cmdb_id, count]
Index: [],   earliest_utc latest_utc
0          NaT        NaT)```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing variables if present; otherwise load files
try:
    metric
    trace
    log
    error_logs
except NameError:
    metric = pd.read_csv('metric.csv')
    trace = pd.read_csv('trace.csv')
    log = pd.read_csv('log.csv')
    error_logs = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetimes (Unix seconds -> utc) to satisfy timezone rule
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Candidate cmdb_ids
candidates = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

# Incident window (UTC)
start = pd.to_datetime("2024-01-17 17:46:36", utc=True)
end   = pd.to_datetime("2024-01-17 18:16:36", utc=True)

# Helper to compute anomalies table for a given dataframe and name-column
def compute_anomalies(df, group_cols, name_col_label):
    # Keep only candidate cmdb_ids
    df_cand = df[df['cmdb_id'].isin(candidates)].copy()
    if df_cand.shape[0] == 0:
        return pd.DataFrame(columns=['cmdb_id', name_col_label, 'total_in_window', 'anomalies_count',
                                     'earliest_anomaly_utc', 'global_p95', 'max_in_window'])
    # Compute global P95 per group using full series (rule 9)
    global_p95 = (
        df_cand
        .groupby(group_cols, as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

    # Filter to incident window (inclusive)
    in_window = df_cand[(df_cand['timestamp'] >= start) & (df_cand['timestamp'] <= end)].copy()

    if in_window.shape[0] == 0:
        # No points in-window at all -> return empty table
        return pd.DataFrame(columns=['cmdb_id', name_col_label, 'total_in_window', 'anomalies_count',
                                     'earliest_anomaly_utc', 'global_p95', 'max_in_window'])

    # Merge global_p95 into in-window rows so we can evaluate anomalies
    merged = in_window.merge(global_p95, on=group_cols, how='left')

    # Determine anomaly boolean per row: value > global_p95
    merged['is_anom'] = merged['value'] > merged['global_p95']

    # Aggregations: total_in_window and max_in_window
    agg_total = (
        merged
        .groupby(group_cols, as_index=False)
        .agg(total_in_window=('value','count'),
             max_in_window=('value','max'))
    )

    # Aggregations for anomalies: count and earliest timestamp
    anom_rows = merged[merged['is_anom']].copy()
    if anom_rows.shape[0] == 0:
        # No anomalies in-window
        return pd.DataFrame(columns=['cmdb_id', name_col_label, 'total_in_window', 'anomalies_count',
                                     'earliest_anomaly_utc', 'global_p95', 'max_in_window'])

    agg_anom = (
        anom_rows
        .groupby(group_cols, as_index=False)
        .agg(anomalies_count=('is_anom','sum'),
             earliest_anomaly_utc=('timestamp','min'))
    )

    # Combine totals, anomalies, and global_p95
    result = (
        agg_total
        .merge(agg_anom, on=group_cols, how='inner')  # inner because we only want groups with anomalies_count>0
        .merge(global_p95, on=group_cols, how='left')
    )

    # Rename columns to match requested output: label the name column accordingly
    # group_cols is like ['cmdb_id','kpi_name'] or ['cmdb_id','trace_name'] etc.
    # Ensure the second column has the requested label
    result = result.rename(columns={group_cols[1]: name_col_label})

    # Reorder columns
    result = result[[
        'cmdb_id',
        name_col_label,
        'total_in_window',
        'anomalies_count',
        'earliest_anomaly_utc',
        'global_p95',
        'max_in_window'
    ]]

    # Sort by anomalies_count desc then max_in_window desc and limit to top 20
    result = result.sort_values(['anomalies_count','max_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)
    return result

# Compute tables
metrics_anoms_table = compute_anomalies(metric, ['cmdb_id','kpi_name'], 'kpi_name')
traces_anoms_table  = compute_anomalies(trace,  ['cmdb_id','trace_name'], 'trace_name')
logs_anoms_table    = compute_anomalies(log,    ['cmdb_id','log_name'],   'log_name')

# Note about error_logs.csv
if error_logs.shape[0] == 0:
    error_logs_note = "error_logs.csv is empty and will be skipped."
else:
    error_logs_note = f"error_logs.csv has {error_logs.shape[0]} rows and will be included if needed."

# Display compact results (variables returned)
metrics_anoms_table, traces_anoms_table, logs_anoms_table, error_logs_note

```
Out[2]:
```
Summary of findings (plain English):

- error_logs.csv is empty and was skipped.

- General pattern: within the incident window (2024-01-17 17:46:36 → 18:16:36 UTC) many candidate services show small numbers of high-value spikes (each reported group had 25 points in-window and anomalies_count = 2, i.e., two samples exceeded the group's global 95th percentile).

- Metrics (groups with anomalies):
  - Memory (mem) spikes in currencyservice, recommendationservice, emailservice, checkoutservice, and productcatalogservice (earliest anomaly times range ~17:49 → 18:09 UTC).
  - Workload spikes in productcatalogservice, currencyservice, recommendationservice, checkoutservice, and emailservice (earliest anomalies ~17:51–17:53 UTC).
  - CPU spikes in currencyservice, productcatalogservice, recommendationservice, checkoutservice, and emailservice (earliest anomalies ~17:51–17:58 UTC).
  - Latency (latency-50 / latency-90) anomalies in checkoutservice, recommendationservice, and productcatalogservice (earliest anomalies around 18:06–18:09 UTC).
  - For every metric row reported: total_in_window = 25, anomalies_count = 2, and the table lists global P95 and the max observed in-window.

- Traces (groups with anomalies):
  - Elevated row_count spikes (trace.*.row_count) appear across productcatalogservice, currencyservice, recommendationservice, and checkoutservice (earliest anomaly times mostly ~17:51–17:53 UTC).
  - A few duration_p95 / duration_mean trace metrics also exceeded their global P95 (checkoutservice and recommendationservice around ~18:02–18:04 UTC).
  - Each reported trace group had 25 points in-window and 2 anomalies.

- Logs (groups with anomalies):
  - log.row_count spikes in currencyservice, recommendationservice, checkoutservice, and emailservice (earliest anomalies ~17:51–17:53 UTC).
  - Each reported log group had 25 points in-window and 2 anomalies.

- Interpretation / takeaway:
  - Multiple candidate services show brief spikes (2 samples per series) during the incident window across memory, workload, CPU, latency, trace row counts, and log row counts. The earliest anomalies for many series cluster around ~17:51 UTC, with some memory and latency anomalies occurring later (~18:04–18:09 UTC).
  - No error_log entries are available to provide additional context.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  total_in_window  anomalies_count      earliest_anomaly_utc    global_p95  max_in_window
0         currencyservice         mem               25                2 2024-01-17 18:04:00+00:00  4.637295e+07   4.669019e+07
1   recommendationservice         mem               25                2 2024-01-17 18:09:00+00:00  4.661820e+07   4.664695e+07
2            emailservice         mem               25                2 2024-01-17 18:06:00+00:00  4.340888e+07   4.342231e+07
3         checkoutservice         mem               25                2 2024-01-17 17:51:00+00:00  1.161445e+07   1.163441e+07
4   productcatalogservice         mem               25                2 2024-01-17 17:49:00+00:00  1.084585e+07   1.113293e+07
5   productcatalogservice    workload               25                2 2024-01-17 17:51:00+00:00  1.470297e+02   1.563078e+02
6         currencyservice    workload               25                2 2024-01-17 17:53:00+00:00  8.102535e+01   8.181323e+01
7         currencyservice         cpu               25                2 2024-01-17 17:51:00+00:00  3.086579e+01   3.095665e+01
8   recommendationservice    workload               25                2 2024-01-17 17:51:00+00:00  1.933422e+01   2.022120e+01
9   productcatalogservice         cpu               25                2 2024-01-17 17:58:00+00:00  3.564950e+00   3.613703e+00
10  recommendationservice         cpu               25                2 2024-01-17 17:51:00+00:00  2.651709e+00   2.666166e+00
11        checkoutservice  latency-90               25                2 2024-01-17 18:06:00+00:00  1.522887e+00   1.726455e+00
12        checkoutservice    workload               25                2 2024-01-17 17:51:00+00:00  1.125040e+00   1.302033e+00
13           emailservice    workload               25                2 2024-01-17 17:51:00+00:00  1.098413e+00   1.283217e+00
14        checkoutservice         cpu               25                2 2024-01-17 17:51:00+00:00  5.081751e-01   5.792772e-01
15        checkoutservice  latency-50               25                2 2024-01-17 18:06:00+00:00  4.918040e-01   5.529882e-01
16           emailservice         cpu               25                2 2024-01-17 17:51:00+00:00  3.394093e-01   3.545136e-01
17  recommendationservice  latency-90               25                2 2024-01-17 18:09:00+00:00  2.360291e-01   2.361048e-01
18  productcatalogservice  latency-90               25                2 2024-01-17 18:09:00+00:00  2.200579e-01   2.201190e-01
19  recommendationservice  latency-50               25                2 2024-01-17 18:09:00+00:00  1.755717e-01   1.756138e-01,                   cmdb_id                                  trace_name  total_in_window  anomalies_count      earliest_anomaly_utc   global_p95  max_in_window
0   productcatalogservice        trace.from_frontendservice.row_count               25                2 2024-01-17 17:51:00+00:00  3807.200000    3909.000000
1         currencyservice        trace.from_frontendservice.row_count               25                2 2024-01-17 17:53:00+00:00  2335.200000    2376.000000
2   productcatalogservice  trace.from_recommendationservice.row_count               25                2 2024-01-17 17:51:00+00:00   588.000000     597.000000
3   recommendationservice        trace.from_frontendservice.row_count               25                2 2024-01-17 17:51:00+00:00   588.800000     597.000000
4   recommendationservice  trace.from_recommendationservice.row_count               25                2 2024-01-17 17:51:00+00:00   588.800000     597.000000
5   recommendationservice    trace.to_productcatalogservice.row_count               25                2 2024-01-17 17:51:00+00:00   588.000000     597.000000
6   recommendationservice    trace.to_recommendationservice.row_count               25                2 2024-01-17 17:51:00+00:00   588.800000     597.000000
7         checkoutservice        trace.from_checkoutservice.row_count               25                2 2024-01-17 17:51:00+00:00   292.000000     295.000000
8         checkoutservice          trace.to_checkoutservice.row_count               25                2 2024-01-17 17:51:00+00:00   292.000000     295.000000
9         checkoutservice    trace.to_productcatalogservice.row_count               25                2 2024-01-17 17:53:00+00:00    85.800000      88.000000
10  productcatalogservice        trace.from_checkoutservice.row_count               25                2 2024-01-17 17:53:00+00:00    85.800000      88.000000
11        checkoutservice        trace.from_frontendservice.row_count               25                2 2024-01-17 17:51:00+00:00    34.600000      35.000000
12        checkoutservice             trace.to_emailservice.row_count               25                2 2024-01-17 17:51:00+00:00    34.600000      35.000000
13        checkoutservice           trace.to_paymentservice.row_count               25                2 2024-01-17 17:51:00+00:00    34.600000      35.000000
14           emailservice        trace.from_checkoutservice.row_count               25                2 2024-01-17 17:51:00+00:00    34.600000      35.000000
15        checkoutservice     trace.from_frontendservice.duration_p95               25                2 2024-01-17 18:03:00+00:00     1.264218       1.292609
16  productcatalogservice                trace.from_root.duration_p95               25                2 2024-01-17 18:04:00+00:00     0.807420       0.810283
17        checkoutservice    trace.from_frontendservice.duration_mean               25                2 2024-01-17 18:02:00+00:00     0.604494       0.624941
18  productcatalogservice               trace.from_root.duration_mean               25                2 2024-01-17 18:07:00+00:00     0.330577       0.332827
19  recommendationservice     trace.from_frontendservice.duration_p95               25                2 2024-01-17 18:02:00+00:00     0.205345       0.205589,                  cmdb_id       log_name  total_in_window  anomalies_count      earliest_anomaly_utc  global_p95  max_in_window
0        currencyservice  log.row_count               25                2 2024-01-17 17:53:00+00:00      2448.6           2495
1  recommendationservice  log.row_count               25                2 2024-01-17 17:51:00+00:00       587.2            597
2        checkoutservice  log.row_count               25                2 2024-01-17 17:51:00+00:00       103.8            105
3           emailservice  log.row_count               25                2 2024-01-17 17:51:00+00:00        34.6             35, 'error_logs.csv is empty and will be skipped.')```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse metric if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Candidate cmdb_ids and incident window (UTC)
candidates = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]
start = pd.to_datetime("2024-01-17 17:46:36", utc=True)
end   = pd.to_datetime("2024-01-17 18:16:36", utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (rule 9)
metric_cand = metric[metric['cmdb_id'].isin(candidates)].copy()
global_p95_metric = (
    metric_cand
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value':'global_p95'})
)

# 2) Filter rows to incident window (inclusive)
in_window = metric_cand[(metric_cand['timestamp'] >= start) & (metric_cand['timestamp'] <= end)].copy()

# 3) For each group compute total_in_window, anomalies_count, earliest anomaly timestamp, max_in_window
if in_window.shape[0] == 0:
    breach_table = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_in_window','anomalies_count','earliest_anomaly_utc',
        'global_p95','max_in_window','breach_pct'
    ])
    top3_anomaly_samples = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp_utc','value'])
else:
    merged = in_window.merge(global_p95_metric, on=['cmdb_id','kpi_name'], how='left')
    # Determine anomalies using global_p95
    merged['is_anom'] = merged['value'] > merged['global_p95']

    agg_total = merged.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        total_in_window=('value','count'),
        max_in_window=('value','max')
    )

    anom_rows = merged[merged['is_anom']].copy()
    if anom_rows.shape[0] == 0:
        breach_table = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','total_in_window','anomalies_count','earliest_anomaly_utc',
            'global_p95','max_in_window','breach_pct'
        ])
        top3_anomaly_samples = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp_utc','value'])
    else:
        agg_anom = anom_rows.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
            anomalies_count=('is_anom','sum'),
            earliest_anomaly_utc=('timestamp','min')
        )

        result = agg_total.merge(agg_anom, on=['cmdb_id','kpi_name'], how='inner') \
                          .merge(global_p95_metric, on=['cmdb_id','kpi_name'], how='left')

        # 1) compute breach_pct = (max_in_window - global_p95)/global_p95*100
        result['breach_pct'] = (result['max_in_window'] - result['global_p95']) / result['global_p95'] * 100.0

        # Round sensibly: breach_pct to 2 decimals, global_p95 and max_in_window to 6 significant digits via round
        result['breach_pct'] = result['breach_pct'].round(2)
        result['global_p95'] = result['global_p95'].round(6)
        result['max_in_window'] = result['max_in_window'].round(6)

        # Keep only groups that had anomalies_count>0 (already ensured by inner merge)
        breach_table = result[[
            'cmdb_id','kpi_name','total_in_window','anomalies_count','earliest_anomaly_utc',
            'global_p95','max_in_window','breach_pct'
        ]].sort_values('breach_pct', ascending=False).head(10).reset_index(drop=True)

        # 2) For top 3 rows, return in-window anomaly samples (value > global_p95), up to 10 rows per group
        top3 = breach_table.head(3)[['cmdb_id','kpi_name']].copy()
        # Merge to get only anomaly rows and limit to top3 groups
        anoms_for_top3 = anom_rows.merge(top3, on=['cmdb_id','kpi_name'], how='inner')
        if anoms_for_top3.shape[0] == 0:
            top3_anomaly_samples = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp_utc','value'])
        else:
            # select columns and sort
            anoms_for_top3 = anoms_for_top3[['cmdb_id','kpi_name','timestamp','value']].copy()
            anoms_for_top3 = anoms_for_top3.rename(columns={'timestamp':'timestamp_utc'})
            anoms_for_top3 = anoms_for_top3.sort_values(['cmdb_id','kpi_name','timestamp_utc'])
            # limit to at most 10 rows per group
            top3_anomaly_samples = anoms_for_top3.groupby(['cmdb_id','kpi_name'], group_keys=False).head(10).reset_index(drop=True)

# Final compact outputs
breach_table, top3_anomaly_samples

```
Out[3]:
```
Summary of results (plain English):

- Analysis focused on five candidate services and used the previously computed global P95 per (cmdb_id, kpi_name). Incident window: 2024-01-17 17:46:36 → 18:16:36 UTC. All timestamps below are UTC.

- Top metric breaches (up to 10 shown). Each row had 25 samples in-window and 2 anomaly samples (value > global P95):

  1) productcatalogservice — kpi: latency-50
     - total_in_window: 25, anomalies_count: 2
     - earliest_anomaly_utc: 2024-01-17 18:04:00+00:00
     - global_p95: 0.069558, max_in_window: 0.082129
     - breach_pct: 18.07%

  2) emailservice — kpi: workload
     - total_in_window: 25, anomalies_count: 2
     - earliest_anomaly_utc: 2024-01-17 17:51:00+00:00
     - global_p95: 1.098413, max_in_window: 1.283217
     - breach_pct: 16.82%

  3) checkoutservice — kpi: workload
     - total_in_window: 25, anomalies_count: 2
     - earliest_anomaly_utc: 2024-01-17 17:51:00+00:00
     - global_p95: 1.125040, max_in_window: 1.302033
     - breach_pct: 15.73%

  (Other groups 4–10 include checkoutservice cpu/latency, productcatalogservice workload/mem, recommendationservice workload, emailservice cpu — all with 2 anomalies and smaller breach_pct.)

- Detailed anomaly samples for the top 3 groups (in-window rows where value > global_P95; at most 10 per group; sorted by cmdb_id,kpi,timestamp):

  - checkoutservice, workload:
    - 2024-01-17 17:51:00+00:00 — 1.302033
    - 2024-01-17 17:54:00+00:00 — 1.144350

  - emailservice, workload:
    - 2024-01-17 17:51:00+00:00 — 1.283217
    - 2024-01-17 17:54:00+00:00 — 1.108867

  - productcatalogservice, latency-50:
    - 2024-01-17 18:04:00+00:00 — 0.070197
    - 2024-01-17 18:13:00+00:00 — 0.082129

- Interpretation: the largest relative breaches (by breach_pct) are a latency spike on productcatalogservice and workload spikes on emailservice and checkoutservice. Each reported group shows just two brief samples above its global P95 during the incident window. Timestamps and values are provided above for investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id    kpi_name  total_in_window  anomalies_count      earliest_anomaly_utc    global_p95  max_in_window  breach_pct
0  productcatalogservice  latency-50               25                2 2024-01-17 18:04:00+00:00  6.955800e-02   8.212900e-02       18.07
1           emailservice    workload               25                2 2024-01-17 17:51:00+00:00  1.098413e+00   1.283217e+00       16.82
2        checkoutservice    workload               25                2 2024-01-17 17:51:00+00:00  1.125040e+00   1.302033e+00       15.73
3        checkoutservice         cpu               25                2 2024-01-17 17:51:00+00:00  5.081750e-01   5.792770e-01       13.99
4        checkoutservice  latency-90               25                2 2024-01-17 18:06:00+00:00  1.522887e+00   1.726455e+00       13.37
5        checkoutservice  latency-50               25                2 2024-01-17 18:06:00+00:00  4.918040e-01   5.529880e-01       12.44
6  productcatalogservice    workload               25                2 2024-01-17 17:51:00+00:00  1.470297e+02   1.563078e+02        6.31
7  recommendationservice    workload               25                2 2024-01-17 17:51:00+00:00  1.933422e+01   2.022120e+01        4.59
8           emailservice         cpu               25                2 2024-01-17 17:51:00+00:00  3.394090e-01   3.545140e-01        4.45
9  productcatalogservice         mem               25                2 2024-01-17 17:49:00+00:00  1.084585e+07   1.113293e+07        2.65,                  cmdb_id    kpi_name             timestamp_utc     value
0        checkoutservice    workload 2024-01-17 17:51:00+00:00  1.302033
1        checkoutservice    workload 2024-01-17 17:54:00+00:00  1.144350
2           emailservice    workload 2024-01-17 17:51:00+00:00  1.283217
3           emailservice    workload 2024-01-17 17:54:00+00:00  1.108867
4  productcatalogservice  latency-50 2024-01-17 18:04:00+00:00  0.070197
5  productcatalogservice  latency-50 2024-01-17 18:13:00+00:00  0.082129)```
```

In [ ]:
# In[4]:

import pandas as pd
import re

# Reuse trace if present, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Parse timestamps to UTC datetimes (Unix seconds -> utc)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Faulty services and incident window
faulty_services = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]
start = pd.to_datetime("2024-01-17 17:46:36", utc=True)
end   = pd.to_datetime("2024-01-17 18:16:36", utc=True)

# 1) Compute global P95 of 'value' for every (cmdb_id, trace_name) using full trace.csv
global_p95_trace = (
    trace
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value':'global_p95'})
)

# 2) Filter trace rows to the incident window (inclusive)
trace_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()

# 3) Select rows where value > global_P95 AND trace_name contains 'trace.to_' or 'trace.from_'
trace_merged = trace_window.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')
mask_dir = trace_merged['trace_name'].str.contains(r'trace\.to_|trace\.from_', regex=True)
mask_anom = trace_merged['value'] > trace_merged['global_p95']
selected = trace_merged[mask_dir & mask_anom].copy()

# 4) Parse trace_name to extract other service and set source/target
# pattern captures direction (to/from) and service name after underscore up to next dot
pat = r'trace\.(to|from)_([^\.]+)\.'
extracted = selected['trace_name'].str.extract(pat)
extracted.columns = ['dir','other_service']
selected = selected.join(extracted)

# Set source and target based on direction
selected['source'] = selected.apply(
    lambda row: row['cmdb_id'] if row['dir']=='to' else row['other_service'], axis=1
)
selected['target'] = selected.apply(
    lambda row: row['other_service'] if row['dir']=='to' else row['cmdb_id'], axis=1
)

# 5) Keep only edges where both source and target are in faulty_services
selected_edges = selected[
    selected['source'].isin(faulty_services) & selected['target'].isin(faulty_services)
].copy()

# 6) Aggregate edges: for each (source,target) produce anomaly_count, earliest_anomaly_utc, max_value_in_window
edges_agg = (
    selected_edges
    .groupby(['source','target'], as_index=False)
    .agg(
        anomaly_count=('value','count'),
        earliest_anomaly_utc=('timestamp','min'),
        max_value_in_window=('value','max')
    )
)

# Sort and limit top 20 as requested
edges_agg_top = edges_agg.sort_values(['anomaly_count','earliest_anomaly_utc'], ascending=[False, True]).head(20).reset_index(drop=True)

# Also produce service-level anomaly counts: sum of anomaly_count where service appears as source or target
if edges_agg_top.shape[0] == 0:
    service_counts = pd.DataFrame(columns=['service','anomaly_count_in_traces'])
else:
    long_counts = pd.concat([
        edges_agg[['source','anomaly_count']].rename(columns={'source':'service'}),
        edges_agg[['target','anomaly_count']].rename(columns={'target':'service'})
    ], ignore_index=True)
    service_counts = long_counts.groupby('service', as_index=False)['anomaly_count'].sum().rename(columns={'anomaly_count':'anomaly_count_in_traces'})
    # Keep only faulty services (include those with zero by reindexing)
    svc_df = pd.DataFrame({'service': faulty_services})
    service_counts = svc_df.merge(service_counts, on='service', how='left').fillna(0)
    service_counts['anomaly_count_in_traces'] = service_counts['anomaly_count_in_traces'].astype(int)
    service_counts = service_counts.sort_values('anomaly_count_in_traces', ascending=False).reset_index(drop=True)

# Compact outputs: edges_agg_top (up to 20) and service_counts
edges_agg_top, service_counts

```
Out[4]:
```
Summary of trace-edge anomalies (trace.csv, timestamps UTC; incident window 2024-01-17 17:46:36 → 18:16:36 UTC):

- Method: selected trace rows in-window where value > group-level global P95 and trace_name contains "trace.to_" or "trace.from_". Parsed each trace_name to form a (source → target) edge and kept edges where both endpoints are in the faulty_services list.

- Top edges (up to 20 shown; sorted by anomaly_count desc then earliest anomaly):
  1. checkoutservice → emailservice — anomaly_count: 12, earliest anomaly: 2024-01-17 17:49:00 UTC, max value: 35.0
  2. checkoutservice → checkoutservice — anomaly_count: 12, earliest anomaly: 2024-01-17 17:51:00 UTC, max value: 295.0
  3. recommendationservice → productcatalogservice — anomaly_count: 12, earliest anomaly: 2024-01-17 17:51:00 UTC, max value: 597.0
  4. recommendationservice → recommendationservice — anomaly_count: 12, earliest anomaly: 2024-01-17 17:51:00 UTC, max value: 597.0
  5. checkoutservice → productcatalogservice — anomaly_count: 12, earliest anomaly: 2024-01-17 17:53:00 UTC, max value: 88.0
  6. checkoutservice → currencyservice — anomaly_count: 10, earliest anomaly: 2024-01-17 17:49:00 UTC, max value: 120.0

- Aggregated anomaly counts per faulty service (counting appearances as source or target):
  - checkoutservice: 58
  - recommendationservice: 36
  - productcatalogservice: 24
  - emailservice: 12
  - currencyservice: 10

Interpretation: checkoutservice is involved in the largest number of anomalous trace edges (many anomalous interactions to emailservice, itself, productcatalogservice, and currencyservice). recommendationservice→productcatalogservice and self-edges for recommendationservice also show high anomaly counts. These edges/times are good starting points for tracing downstream effects or investigating service-to-service latency/throughput spikes during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(                  source                 target  anomaly_count      earliest_anomaly_utc  max_value_in_window
0        checkoutservice           emailservice             12 2024-01-17 17:49:00+00:00                 35.0
1        checkoutservice        checkoutservice             12 2024-01-17 17:51:00+00:00                295.0
2  recommendationservice  productcatalogservice             12 2024-01-17 17:51:00+00:00                597.0
3  recommendationservice  recommendationservice             12 2024-01-17 17:51:00+00:00                597.0
4        checkoutservice  productcatalogservice             12 2024-01-17 17:53:00+00:00                 88.0
5        checkoutservice        currencyservice             10 2024-01-17 17:49:00+00:00                120.0,                  service  anomaly_count_in_traces
0        checkoutservice                       58
1  recommendationservice                       36
2  productcatalogservice                       24
3           emailservice                       12
4        currencyservice                       10)```
```